# Back-to-Back Causal Conv1D Backward

This notebook tests gradients for the **fused back-to-back causal conv1d** using cuDNN.

The `cudnn.ops.b2b_causal_conv1d` API integrates with `torch.autograd`, so backward is handled
automatically via `.backward()`. The fused B2B backward kernel computes gradients flowing
through the post-gated final output `y_gated`:

```python
y_gated = cudnn.ops.b2b_causal_conv1d(x, weights_proj, weights_mixer, skip_bias)
loss = y_gated.sum()
loss.backward()
```

and compare the resulting gradients against a decomposed PyTorch reference:

- $dx$ — gradient w.r.t. input, shape `(batch, 3*dim, seq_len)`
- $dw_{proj}$ — gradient w.r.t. projection weights, shape `(3*dim, K_{proj})`
- $dw_{mixer}$ — gradient w.r.t. mixer weights, shape `(dim, K_{mixer})`
- $db_{skip}$ — gradient w.r.t. skip bias, shape `(dim,)`

All tensors use FP32 for numerical stability.

## Prerequisites and Setup
This notebook requires an NVIDIA GPU (Hopper or later recommended) and cuDNN 9.24.0 or later.

**Environment setup** — make sure the cuDNN runtime library and the `cudnn` Python package are discoverable before launching the notebook:

- **Option A – pip install:**
  ```bash
  pip install nvidia-cudnn-frontend
  ```
- **Option B – set paths manually:**
  ```bash
  export LD_LIBRARY_PATH=/path/to/cudnn/lib:${LD_LIBRARY_PATH}
  export PYTHONPATH=/path/to/cudnn_frontend/build_python:${PYTHONPATH}
  ```

Adjust the paths above to match your local build or installation directory.

In [1]:
# !nvidia-smi

In [2]:
# !pip install nvidia-cudnn-cu12
# !pip install nvidia-cudnn-frontend
# !pip3 install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu128

## Overview

We will test backward gradients for a B2B causal conv1d with:

- batch size: 2
- dim (channels): 64
- sequence length: 512
- projection kernel size: 4
- mixer kernel size: 7
- data type: Float32

We compare the cuDNN autograd result against a decomposed PyTorch reference.

In [3]:
import torch
import torch.nn.functional as F
import cudnn

print("cuDNN backend version:", cudnn.backend_version())

cuDNN backend version: 92300


In [4]:
def causal_conv1d_ref(x, weight):
    """Depthwise causal conv1d.

    Args:
        x:      (batch, channels, seq_len)
        weight: (channels, kernel_size)
    """
    x_padded = F.pad(x, (weight.shape[1] - 1, 0))
    return F.conv1d(x_padded, weight.unsqueeze(1), groups=x.shape[1])


def b2b_causal_conv1d_ref(x, weights_proj, weights_mixer, skip_bias):
    """Reference: decomposed B2B causal conv1d.

    Returns the post-gated final output ``y_gated``.

    Args:
        x:             (batch, 3*dim, seq_len)
        weights_proj:  (3*dim, K_proj)
        weights_mixer: (dim, K_mixer)
        skip_bias:     (dim,)
    """
    proj = causal_conv1d_ref(x, weights_proj)
    gated = proj[:, 1::3, :] * proj[:, 2::3, :]
    y = causal_conv1d_ref(gated, weights_mixer) + skip_bias[:, None] * gated
    y_gated = y * proj[:, 0::3, :]
    return y_gated

In [5]:
batch = 2
dim = 64
seq_len = 512
k_proj = 4
k_mixer = 7
dtype = torch.float32

has_cuda = torch.cuda.is_available()

torch.manual_seed(42)

x = torch.randn(batch, 3 * dim, seq_len, dtype=dtype)
weights_proj = torch.randn(3 * dim, k_proj, dtype=dtype)
weights_mixer = torch.randn(dim, k_mixer, dtype=dtype)
skip_bias = torch.randn(dim, dtype=dtype)

print(f"CUDA available: {has_cuda}")
print(f"x:             {x.shape}, dtype={x.dtype}")
print(f"weights_proj:  {weights_proj.shape}, dtype={weights_proj.dtype}")
print(f"weights_mixer: {weights_mixer.shape}, dtype={weights_mixer.dtype}")
print(f"skip_bias:     {skip_bias.shape}, dtype={skip_bias.dtype}")

CUDA available: True
x:             torch.Size([2, 192, 512]), dtype=torch.float32
weights_proj:  torch.Size([192, 4]), dtype=torch.float32
weights_mixer: torch.Size([64, 7]), dtype=torch.float32
skip_bias:     torch.Size([64]), dtype=torch.float32


In [6]:
x_ref = x.clone().requires_grad_(True)
wp_ref = weights_proj.clone().requires_grad_(True)
wm_ref = weights_mixer.clone().requires_grad_(True)
sb_ref = skip_bias.clone().requires_grad_(True)

y_gated_ref = b2b_causal_conv1d_ref(x_ref, wp_ref, wm_ref, sb_ref)
loss_ref = y_gated_ref.sum()
loss_ref.backward()

print(f"dx_ref:             {x_ref.grad.shape}")
print(f"dweights_proj_ref:  {wp_ref.grad.shape}")
print(f"dweights_mixer_ref: {wm_ref.grad.shape}")
print(f"dskip_bias_ref:     {sb_ref.grad.shape}")

dx_ref:             torch.Size([2, 192, 512])
dweights_proj_ref:  torch.Size([192, 4])
dweights_mixer_ref: torch.Size([64, 7])
dskip_bias_ref:     torch.Size([64])


In [7]:
if has_cuda:
    x_gpu = x.cuda().requires_grad_(True)
    wp_gpu = weights_proj.cuda().requires_grad_(True)
    wm_gpu = weights_mixer.cuda().requires_grad_(True)
    sb_gpu = skip_bias.cuda().requires_grad_(True)

    y_gated_cudnn = cudnn.ops.b2b_causal_conv1d(x_gpu, wp_gpu, wm_gpu, sb_gpu)
    loss_cudnn = y_gated_cudnn.sum()
    loss_cudnn.backward()

    print(f"dx_cudnn:             {x_gpu.grad.shape}")
    print(f"dweights_proj_cudnn:  {wp_gpu.grad.shape}")
    print(f"dweights_mixer_cudnn: {wm_gpu.grad.shape}")
    print(f"dskip_bias_cudnn:     {sb_gpu.grad.shape}")
else:
    print("Skipping cuDNN backward (no CUDA device).")

dx_cudnn:             torch.Size([2, 192, 512])
dweights_proj_cudnn:  torch.Size([192, 4])
dweights_mixer_cudnn: torch.Size([64, 7])
dskip_bias_cudnn:     torch.Size([64])


In [8]:
if has_cuda:
    atol = 1e-3

    for name, cudnn_grad, ref_grad in [
        ("dx",             x_gpu.grad.cpu(),  x_ref.grad),
        ("dweights_proj",  wp_gpu.grad.cpu(), wp_ref.grad),
        ("dweights_mixer", wm_gpu.grad.cpu(), wm_ref.grad),
        ("dskip_bias",     sb_gpu.grad.cpu(), sb_ref.grad),
    ]:
        max_abs = (cudnn_grad - ref_grad).abs().max().item()
        print(f"{name:16s} max_abs_diff={max_abs:.4e}")
        assert max_abs < atol, f"{name} verification failed: max_abs={max_abs}"

    print("\nPASSED: cuDNN B2B causal_conv1d backward matches reference.")
else:
    print("Skipping verification (no CUDA device).")

dx               max_abs_diff=9.1553e-05
dweights_proj    max_abs_diff=4.8828e-04
dweights_mixer   max_abs_diff=2.4414e-04
dskip_bias       max_abs_diff=1.2207e-04

PASSED: cuDNN B2B causal_conv1d backward matches reference.
